# Gráfico Dinâmico de Epidemia
### Modelo de análise: SIRD (Susceptible, Infected, Recovered, Deceased)
### Epidemia: Ebola
### País: Guiné
### Período: 2014-2016

---
# Organização dos Dados

## Dados brutos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import copy
import numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from statsmodels.nonparametric.smoothers_lowess import lowess

habitantes=12600000

df= pd.read_csv('data/ebola/Africa/Guine_SIRD.csv')
print('Número de linhas do csv:', len(df))
dados = df.to_numpy().tolist()

print('\nDados brutos do csv:\n')
for linha in dados:
    print(linha)

## Dados para o modelo SIRD

In [ ]:
x = [habitantes,0,0,0]
historico = [x]

print('\nDados para o modelo SIRD:\n')
print(historico[0])

for linha in dados:
    linha_sird = [linha[1], linha[2], linha[3], linha[4]]
    historico.append(linha_sird)
    print(linha_sird)

## Dados SIRD normalizados

In [ ]:
print('\nDados SIRD normalizados:\n')
historico_normalizado = copy.deepcopy(historico)
for linha in historico_normalizado:
    for i in range(4):
        linha[i]/=habitantes
    print(linha)

## Dados dos parâmetros IR

In [ ]:
print('\nDados IR:\n')
historico_IR = []
for linha in historico:
    linha_ir=[linha[1], linha[2]]
    historico_IR.append(linha_ir)
    print(linha_ir)

## Obtendo o maior dado para a normalização

In [ ]:
maior=0
for linha in historico_IR:
    for i in range(2):
        if(linha[i]>maior):
            maior=linha[i]
print('\nMaior dado dos parâmetros IR:', maior)

## Dados IR normalizados

In [ ]:
historico_IR_normalizados = copy.deepcopy(historico_IR)

print('\nDados IR normalizados:\n')
for linha in historico_IR_normalizados:
    for i in range(2):
        linha[i]/=maior
    print(linha)

# Plotando o Gráfico

In [ ]:
n=len(historico_normalizado)
dias=list(range(n))

S=[h[0] for h in historico_normalizado]
I=[h[1] for h in historico_normalizado]
R=[h[2] for h in historico_normalizado]
D=[h[3] for h in historico_normalizado]

fig, ax = plt.subplots(figsize=(10,6))
ax.set_xlim(0, n-1)
ax.set_yscale('log')
ax.set_ylim(1e-6, 1.30)
ax.set_xlabel('Dias')
ax.set_ylabel('Proporção da População (log)')
ax.set_title('Modelo SIRD - Epidemia do Ebola em Guiné (2014-2016)')
ax.grid(True, linestyle='--', alpha=0.5)

line_s, = ax.plot([], [], label='Suscetíveis (S)', color='steelblue', lw=2)
line_i, = ax.plot([], [], label='Infectados (I)', color='firebrick', lw=2)
line_r, = ax.plot([], [], label='Recuperados (R)', color='mediumseagreen', lw=2)
line_d, = ax.plot([], [], label='Mortos (D)', color='dimgray', lw=2)
ax.legend(loc='center right')

texto_dia = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=10)

def init():
    line_s.set_data([], [])
    line_i.set_data([], [])
    line_r.set_data([], [])
    line_d.set_data([], [])
    texto_dia.set_text('')
    
    return line_s, line_i, line_r, line_d, texto_dia

def update(frame):
    x = dias[:frame+1]
    line_s.set_data(x, S[:frame+1])
    line_i.set_data(x, I[:frame+1])
    line_r.set_data(x, R[:frame+1])
    line_d.set_data(x, D[:frame+1])
    texto_dia.set_text(f'Dia {frame}')
    
    return line_s, line_i, line_r, line_d, texto_dia

animation = FuncAnimation(fig, update, frames=n, init_func=init, blit=False, interval=80, repeat=True)

plt.close()
HTML(animation.to_jshtml())

## Observações
- O gráfico acima está em escala logarítmica devido a grande diferença do número de casos em relação ao número de habitantes de Guiné;

- No dataset original, existem 4 classificações de casos: confirmados, suspeitos, prováveis e totais. Porém, apenas os casos totais são necessários para a criação do gráfico. Além disso, eles são a única classificação possível de analisar, pois as outras apresentam inconsistências ao longo das páginas da planilha e servem para outros tipos de análises.

---
# Gráfico dos parâmetros IR (Escala logarítmica)

In [ ]:
n=len(historico_normalizado)
dias=list(range(n))

I=[h[1] for h in historico_normalizado]
R=[h[2] for h in historico_normalizado]

fig, ax = plt.subplots(figsize=(10,6))
ax.set_xlim(0, n-1)
ax.set_yscale('log')
ax.set_ylim(1e-6, 1e-2)
ax.set_xlabel('Dias')
ax.set_ylabel('Proporção da População (log)')
ax.set_title('Modelo SIRD (Convertido em IR) - Epidemia do Ebola em Guiné (2014-2016)')
ax.grid(True, linestyle='--', alpha=0.5)

line_i, = ax.plot([], [], label='Infectados (I)', color='firebrick', lw=2)
line_r, = ax.plot([], [], label='Recuperados (R)', color='mediumseagreen', lw=2)
ax.legend(loc='upper right')

texto_dia = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=10)

def init():
    line_i.set_data([], [])
    line_r.set_data([], [])
    texto_dia.set_text('')
    
    return line_i, line_r, texto_dia

def update(frame):
    x = dias[:frame+1]
    line_i.set_data(x, I[:frame+1])
    line_r.set_data(x, R[:frame+1])
    texto_dia.set_text(f'Dia {frame}')
    
    return line_i, line_r, texto_dia

animation = FuncAnimation(fig, update, frames=n, init_func=init, blit=False, interval=80, repeat=True)

plt.close()
HTML(animation.to_jshtml())

# Gráfico dos parâmetros IR (Porcentagem)

In [ ]:
n=len(historico_IR_normalizados)
dias=list(range(n))

I=[h[0] for h in historico_IR_normalizados]
R=[h[1] for h in historico_IR_normalizados]

fig, ax = plt.subplots(figsize=(10,6))
ax.set_xlim(0, n-1)
ax.set_ylim(0, 1)
ax.set_xlabel('Dias')
ax.set_ylabel('Proporção da População (%)')
ax.set_title('Modelo SIRD (Convertido em IR) - Epidemia do Ebola em Guiné (2014-2016)')
ax.grid(True, linestyle='--', alpha=0.5)

line_i, = ax.plot([], [], label='Infectados (I)', color='firebrick', lw=2)
line_r, = ax.plot([], [], label='Recuperados (R)', color='mediumseagreen', lw=2)
ax.legend(loc='center right')

texto_dia = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=10)

def init():
    line_i.set_data([], [])
    line_r.set_data([], [])
    texto_dia.set_text('')
    
    return line_i, line_r, texto_dia

def update(frame):
    x = dias[:frame+1]
    line_i.set_data(x, I[:frame+1])
    line_r.set_data(x, R[:frame+1])
    texto_dia.set_text(f'Dia {frame}')
    
    return line_i, line_r, texto_dia

animation = FuncAnimation(fig, update, frames=n, init_func=init, blit=False, interval=80, repeat=True)

plt.close()
HTML(animation.to_jshtml())

# Gráfico IR suavizado

In [ ]:
n=len(historico_IR_normalizados)
dias=list(range(n))

I=[h[0] for h in historico_IR_normalizados]
R=[h[1] for h in historico_IR_normalizados]

frac = 0.15

I_smooth = lowess(I, dias, frac=frac, return_sorted=True)
R_smooth = lowess(R, dias, frac=frac, return_sorted=True)

I_lowess = I_smooth[:, 1]
R_lowess = R_smooth[:, 1]

fig, ax = plt.subplots(figsize=(10,6))
ax.set_xlim(0, n-1)
ax.set_ylim(0, 1)
ax.set_xlabel('Dias')
ax.set_ylabel('Proporção da População (%)')
ax.set_title('Modelo SIRD (IR) - Epidemia do Ebola em Guiné (2014-2016) - Suavizado')
ax.grid(True, linestyle='--', alpha=0.5)

line_i, = ax.plot([], [], label='Infectados (I)', color='firebrick', lw=2)
line_r, = ax.plot([], [], label='Recuperados (R)', color='mediumseagreen', lw=2)
ax.legend(loc='center right')

texto_dia = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=10)

def init():
    line_i.set_data([], [])
    line_r.set_data([], [])
    texto_dia.set_text('')
    
    return line_i, line_r, texto_dia

def update(frame):
    x = dias[:frame+1]
    line_i.set_data(x, I_lowess[:frame+1])
    line_r.set_data(x, R_lowess[:frame+1])
    texto_dia.set_text(f'Dia {frame}')
    
    return line_i, line_r, texto_dia

animation = FuncAnimation(fig, update, frames=n, init_func=init, blit=False, interval=80, repeat=True)

plt.close()
HTML(animation.to_jshtml())

# Gráfico SIRD suavizado

In [ ]:
n=len(historico_normalizado)
dias=list(range(n))

S=[h[0] for h in historico_normalizado]
I=[h[1] for h in historico_normalizado]
R=[h[2] for h in historico_normalizado]
D=[h[3] for h in historico_normalizado]

frac = 0.15

S_smooth = lowess(S, dias, frac=frac, return_sorted=True)
I_smooth = lowess(I, dias, frac=frac, return_sorted=True)
R_smooth = lowess(R, dias, frac=frac, return_sorted=True)
D_smooth = lowess(D, dias, frac=frac, return_sorted=True)

S_lowess = S_smooth[:, 1]
I_lowess = I_smooth[:, 1]
R_lowess = R_smooth[:, 1]
D_lowess = D_smooth[:, 1]

maior = 0
for valor in S_lowess:
    if valor > maior:
        maior = valor

S_normal_lowess = []
I_normal_lowess = []
R_normal_lowess = []
D_normal_lowess = []

for x, y, z, w in zip(S_lowess, I_lowess, R_lowess, D_lowess):
    S_normal_lowess.append(x/maior)
    I_normal_lowess.append(y/maior)
    R_normal_lowess.append(z/maior)
    D_normal_lowess.append(w/maior)

fig, ax = plt.subplots(figsize=(10,6))
ax.set_xlim(0, n-1)
ax.set_yscale('log')
ax.set_ylim(1e-6, 1.30)
ax.set_xlabel('Dias')
ax.set_ylabel('Proporção da População (log)')
ax.set_title('Modelo SIRD - Epidemia do Ebola em Guiné (2014-2016) - Suavizado')
ax.grid(True, linestyle='--', alpha=0.5)

line_s, = ax.plot([], [], label='Suscetíveis (S)', color='steelblue', lw=2)
line_i, = ax.plot([], [], label='Infectados (I)', color='firebrick', lw=2)
line_r, = ax.plot([], [], label='Recuperados (R)', color='mediumseagreen', lw=2)
line_d, = ax.plot([], [], label='Mortos (D)', color='dimgray', lw=2)
ax.legend(loc='center right')

texto_dia = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=10)

def init():
    line_s.set_data([], [])
    line_i.set_data([], [])
    line_r.set_data([], [])
    line_d.set_data([], [])
    texto_dia.set_text('')
    
    return line_s, line_i, line_r, line_d, texto_dia

def update(frame):
    x = dias[:frame+1]
    line_s.set_data(x, S_normal_lowess[:frame+1])
    line_i.set_data(x, I_normal_lowess[:frame+1])
    line_r.set_data(x, R_normal_lowess[:frame+1])
    line_d.set_data(x, D_normal_lowess[:frame+1])
    texto_dia.set_text(f'Dia {frame}')
    
    return line_s, line_i, line_r, line_d, texto_dia

animation = FuncAnimation(fig, update, frames=n, init_func=init, blit=False, interval=80, repeat=True)

plt.close()
HTML(animation.to_jshtml())

## Encontrando a Matriz de Transição aproximada do gráfico acima

In [ ]:
from scipy.optimize import nnls

X_completo = np.array([S_normal_lowess, I_normal_lowess, R_normal_lowess, D_normal_lowess])
dia_pico = np.argmax(I_normal_lowess)
dia_pico+=30

X   = X_completo[:, :-1]
X_l = X_completo[:, 1:]

X_growth = X[:, :dia_pico]
X_l_growth = X_l[:, :dia_pico]

print('Matriz X:\n', X)
print("\nMatriz X':\n", X_l)

mascara = np.array([
    [True,  False, False, False],
    [True,  True,  False, False],
    [False, True,  True,  False],
    [False, True,  False, True ],
])

n_estados = X_growth.shape[0]
A = np.zeros((n_estados, n_estados))

for i in range(n_estados):
    colunas_permitidas = np.where(mascara[i])[0]
    a_reduzida = X_growth[colunas_permitidas, :].T
    b_linha = X_l_growth[i, :]

    coeficientes = nnls(a_reduzida, b_linha)[0]

    A[i, colunas_permitidas] = coeficientes

somas_colunas = A.sum(axis=0)
fator = np.maximum(somas_colunas, 1)
A = A/fator

print("\nMatriz A:\n", A)

# Validando a Matriz

In [ ]:
dias = 94
x = X[:, 0]
historico = [x]

for t in range(dias):
    x = A @ x
    historico.append(x)
historico = np.array(historico)

fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, len(historico)-1)
ax.set_yscale('log')
ax.set_ylim(1e-6, 1e-2)
ax.set_xlabel('Dias')
ax.set_ylabel('Proporção da População (log)')
ax.set_title('Testando a matriz A')
ax.grid(True, linestyle='--', alpha=0.5)

cores = ['firebrick', 'mediumseagreen', 'dimgray'] 
labels = ['Infectados', 'Recuperados', 'Falecidos']
linhas = [ax.plot([], [], lw=2, color=cores[i], label=labels[i])[0] for i in range(3)]
ax.legend(loc='upper right')

def init():
    for i in range(3):
        linhas[i].set_data([], [])
    return linhas

def atualizar(frame):
    for i in range(3):
        linhas[i].set_data(list(range(frame+1)), historico[:frame+1, i+1])
    return linhas

animation = FuncAnimation(fig, atualizar, frames=len(historico), init_func=init, blit=False, interval=80, repeat=True)

plt.close()
HTML(animation.to_jshtml())

### Observação
- O gráfico gerado pela matriz **A** ainda não está fiel ao gráfico do SIRD com regressão